### Daten erstellen und customclassifier fitten

In [37]:
from utils import create_train_test_data, create_new_dataset_with_ipcw_weights, ipc_weighted_mse
from class_DecisionTreeBaggingClassifier import DecisionTreeBaggingClassifier
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter

n = 1000
seed = 42
tau = 32
B_RF = 2000

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': seed, 'tau': tau}


df_train, df_test\
    , n_events_after_cut_train, portion_censored_after_cut_train\
    , n_events_after_cut_test, portion_censored_after_cut_test = create_train_test_data(params=params_data_creation)



X_pred_point = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})



In [36]:
params_rf = {  'n_estimators':2000,                        
                'max_depth':4,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  seed,
                'weighted_bootstrapping':     True,  }

# Fitten des Random Forest Modells
params_rf["random_state"] = seed
clf = DecisionTreeBaggingClassifier(params_rf)
clf.fit(
    X=df_train.drop(
        ["time", "event", "weights_ipcw", "survived"], axis=1
    ).values,
    y=df_train["survived"].values,
    sample_weights=df_train["weights_ipcw"].values,
)

_ , pred  =clf.predict_proba(df_test.drop(
            ["weights_ipcw", "survived", "time", "event"], axis=1
        ).values)

# Evaluation auf Testdaten
rf_mse_ipcw = ipc_weighted_mse(
    y_true=df_test["survived"].values,
    y_pred=pred,
    sample_weight=df_test["weights_ipcw"].values,
)
rf_mse_ipcw

0.14510979216333666